In [ ]:
# This cell is theoretical fluxonium model, which will provide data for training ML model
import numpy as np
from scipy.sparse import diags, csc_matrix, identity, kron
from scipy.sparse.linalg import eigsh


def _fluxonium_basis(EC, EL, EJ, flux_ext, levels=8, grid=2000, flux_max=12*np.pi):
    """Creat Fluxonium Hamiltonian, return (E, n_Q projected into eigenbasis) using FD in phase."""
    
    flux = np.linspace(-flux_max, flux_max, grid)
    dflux = flux[1] - flux[0]
    main, off = 2.0*np.ones(grid), -1.0*np.ones(grid-1)
    lap = diags([off, main, off], offsets=[-1, 0, 1]) / (dflux**2)
    T = -4.0 * EC * lap
    V = 0.5*EL*(flux**2) - EJ*np.cos(flux-flux_ext)
    H = T + diags(V, 0)

    evals, evecs = eigsh(H, k=levels, which="SA")
    idx = np.argsort(evals)
    E = np.array(evals[idx])
    Psi = np.array(evecs[:, idx])

    offp =  np.ones(grid-1) / (2.0 * dflux)
    offm = -np.ones(grid-1) / (2.0 * dflux)
    d1 = diags([offm, np.zeros(grid), offp], [-1, 0, 1], dtype=np.complex128)
    n_grid = (-1j) * d1
    nQ = Psi.conj().T @ (n_grid @ Psi)
    return E, nQ


def _resonator_ops_from_fr(fr_bare, N_phot=6):
    """Creat Resonator Hamiltonian, return H_res (GHz), n_R = i(a†-a), N = a†a in N_phot Fock basis."""
    
    a = np.zeros((N_phot, N_phot), dtype=complex)
    for n in range(1, N_phot):
        a[n-1, n] = np.sqrt(n)
    adag = a.conj().T
    Hres = fr_bare * (adag @ a + 0.5 * np.eye(N_phot))
    nR = 1j * (adag - a)
    Nph = adag @ a
    return csc_matrix(Hres), csc_matrix(nR), csc_matrix(Nph)


def _build_total_H(EC, EL, EJ, flux_ext, fr_bare, g, q_levels=8, N_phot=6, grid=2000, flux_max=12*np.pi):
    EQ, nQ = _fluxonium_basis(EC, EL, EJ, flux_ext, levels=q_levels, grid=grid, flux_max=flux_max)
    Hq = diags(EQ, 0, format='csc')
    Iq = identity(q_levels, format='csc')
    Hres, nR, Nph = _resonator_ops_from_fr(fr_bare, N_phot=N_phot)
    Ir = identity(N_phot, format='csc')
    Hint = g * kron(csc_matrix(nQ), nR)
    Htot = kron(Hq, Ir) + kron(Iq, Hres) + Hint
    return Htot, Nph, q_levels, N_phot


def _pick_state(E, V, Pg_full, Pe_full, Nph_full, manifold, n_target):
    """Select the eigenstate closest to |manifold, n_target>, since they might not be lowest 4 eigenstates.
    For g/e state of qubit, expection value of P_g/P_e projector should close to 1. For 0/1 state of resonator expectation of N operator should close to 0/1"""
    
    P = Pg_full if manifold == "g" else Pe_full
    best_E = None
    best_score = np.inf
    for j in range(E.size):
        v = V[:, j]
        p = float(np.real_if_close(v.conj().T @ (P @ v)))
        nbar = float(np.real_if_close(v.conj().T @ (Nph_full @ v)))
        score = (1.0 - p) + abs(nbar - n_target)
        if score < best_score:
            best_score = score
            best_E = E[j]
    return best_E


def _readout_frequencies(EC, EL, EJ, flux_ext, fr, g, q_levels=10, N_phot=6, grid=2000, flux_max=12*np.pi):
    """Simulate frequencies fr_g and fr_e of specific flux point"""
    
    Htot, Nph, Lq, Nr = _build_total_H(EC, EL, EJ, flux_ext, fr, g, q_levels=q_levels,
        N_phot=N_phot, grid=grid, flux_max=flux_max)
    k_eval = min(4 * Lq, Htot.shape[0] - 2) # Make sure to include g0, g1, e0, e1
    evals, evecs = eigsh(Htot, k=k_eval, which="SA")
    idx = np.argsort(evals)
    E = evals[idx]
    V = evecs[:, idx]

    Pg = np.zeros((Lq, Lq))
    Pg[0, 0] = 1.0
    Pe = np.zeros((Lq, Lq))
    Pe[1, 1] = 1.0
    Pg_full = kron(csc_matrix(Pg), identity(Nr, format="csc"))
    Pe_full = kron(csc_matrix(Pe), identity(Nr, format="csc"))
    Nph_full = kron(identity(Lq, format="csc"), Nph)

    Eg0 = _pick_state(E, V, Pg_full, Pe_full, Nph_full, manifold="g", n_target=0)
    Eg1 = _pick_state(E, V, Pg_full, Pe_full, Nph_full, manifold="g", n_target=1)
    Ee0 = _pick_state(E, V, Pg_full, Pe_full, Nph_full, manifold="e", n_target=0)
    Ee1 = _pick_state(E, V, Pg_full, Pe_full, Nph_full, manifold="e", n_target=1)
    fr_g = Eg1 - Eg0
    fr_e = Ee1 - Ee0
    return fr_g, fr_e

In [ ]:
import math
import os
import time
from dataclasses import dataclass
from typing import Tuple, Optional
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# Here those values below are just default values that will be changed in next cell
#Just brief explanation here do not need to change those
@dataclass
class GenerateConfig:
    EC0: float = 1.0
    EL0: float = 0.8
    EJ0: float = 3.0
    g0: float = 0.3
    fr0: float = 5.1
    EC_span: float = 0.10   # ±10%
    EL_span: float = 0.10   # ±10%
    EJ_span: float = 0.50   # ±50%
    g_span: float = 1.0    # ±100%
    fr_span: float = 0.05   # ±5%

    # expensive base curves total ~ 500
    n_axis_points: int = 16   # 5 * 16 = 80
    n_joint_random: int = 420 # 80 + 420 = 500

    # flux grid
    n_flux_points: int = 121 # Your CNN values (number of flux points you pick evenly in a period)
    flux_periods: float = 1.0
    flux_start_periods: float = 0.0

    # cheap augmentation by circular rolling
    n_offset_steps: int = 31
    add_noise_std: float = 0.0

    # numerics
    q_levels: int = 10
    N_phot: int = 6
    grid: int = 2000
    flux_max: float = 12 * np.pi

    seed: int = 1234
    save_path: str = "fluxonium_better_local_dataset.npz" # Path of saving this ML model


# Dataset generation helpers
def generate_flux_grid(cfg: GenerateConfig) -> np.ndarray:
    return np.linspace(
        cfg.flux_start_periods,
        cfg.flux_start_periods + cfg.flux_periods,
        cfg.n_flux_points,
        endpoint=False)


def period_to_phase(period_value: np.ndarray) -> np.ndarray:
    return 2.0 * np.pi * period_value


def generate_curve(EC, EL, EJ, fr, g, flux_base_periods, cfg: GenerateConfig) -> np.ndarray:
    flux_phase = period_to_phase(flux_base_periods)
    curve = np.empty(len(flux_phase), dtype=np.float64)

    for i, flux_ext in enumerate(flux_phase):
        fr_g, _ = _readout_frequencies(EC,EL,EJ,flux_ext,fr,g,cfg.q_levels,cfg.N_phot,cfg.grid,cfg.flux_max)
        curve[i] = float(np.real_if_close(fr_g))
    return curve


def roll_with_offsets(curve: np.ndarray, n_offset_steps: int):
    n = len(curve)
    if n_offset_steps <= 1:
        return [curve.astype(np.float32)], [0], [0.0]

    step_indices = np.linspace(0, n - 1, n_offset_steps, dtype=int)
    step_indices = np.unique(step_indices)
    curves, steps, offset_fracs = [], [], []
    for s in step_indices:
        curves.append(np.roll(curve, s).astype(np.float32))
        steps.append(int(s))
        offset_fracs.append(float(s / n))
    return curves, steps, offset_fracs


def build_better_local_dataset(cfg: GenerateConfig):
    rng = np.random.default_rng(cfg.seed)
    center = {
        "EC": cfg.EC0,
        "EL": cfg.EL0,
        "EJ": cfg.EJ0,
        "g": cfg.g0,
        "fr": cfg.fr0,
    }

    flux_base = generate_flux_grid(cfg).astype(np.float32)
    X_list = []
    y_list = []
    source_list = []
    offset_step_list = []

    swept_names = ["EC", "EL", "EJ", "g", "fr"]
    total_base = 5 * cfg.n_axis_points + cfg.n_joint_random
    t0 = time.time()
    done = 0
    print("Starting improved dataset generation...")
    print(f"Total expensive base curves: {total_base}")
    print(f"Axis points per parameter: {cfg.n_axis_points}")
    print(f"Joint random base curves: {cfg.n_joint_random}")
    print(f"Offset rolls per base curve: {cfg.n_offset_steps}")

    span_map = {
        "EC": cfg.EC_span,
        "EL": cfg.EL_span,
        "EJ": cfg.EJ_span,
        "g": cfg.g_span,
        "fr": cfg.fr_span,
    }

    for pname in swept_names:
        c = center[pname]
        span = span_map[pname]
        vals = np.linspace(c * (1 - span), c * (1 + span), cfg.n_axis_points)

        for val in vals:
            pars = center.copy()
            pars[pname] = float(val)

            curve = generate_curve(
                EC=pars["EC"],
                EL=pars["EL"],
                EJ=pars["EJ"],
                fr=pars["fr"],
                g=pars["g"],
                flux_base_periods=flux_base,
                cfg=cfg,
            )

            rolled_curves, offset_steps, offset_fracs = roll_with_offsets(curve, cfg.n_offset_steps)

            for x, step, off in zip(rolled_curves, offset_steps, offset_fracs):
                if cfg.add_noise_std > 0:
                    x = x + rng.normal(0.0, cfg.add_noise_std, size=x.shape)
                X_list.append(x.astype(np.float32))
                y_list.append([pars["EC"], pars["EL"], pars["EJ"], pars["g"], pars["fr"], off])
                source_list.append("axis")
                offset_step_list.append(step)

            done += 1
            if done % 10 == 0 or done == total_base:
                print(f"[{done}/{total_base}] elapsed = {time.time() - t0:.1f} s")

    for _ in range(cfg.n_joint_random):
        pars = {
            "EC": cfg.EC0 * (1 + rng.uniform(-cfg.EC_span, cfg.EC_span)),
            "EL": cfg.EL0 * (1 + rng.uniform(-cfg.EL_span, cfg.EL_span)),
            "EJ": cfg.EJ0 * (1 + rng.uniform(-cfg.EJ_span, cfg.EJ_span)),
            "g":  cfg.g0  * (1 + rng.uniform(-cfg.g_span,  cfg.g_span)),
            "fr": cfg.fr0 * (1 + rng.uniform(-cfg.fr_span, cfg.fr_span)),
        }

        curve = generate_curve(
            EC=pars["EC"],
            EL=pars["EL"],
            EJ=pars["EJ"],
            fr=pars["fr"],
            g=pars["g"],
            flux_base_periods=flux_base,
            cfg=cfg,
        )

        rolled_curves, offset_steps, offset_fracs = roll_with_offsets(curve, cfg.n_offset_steps)
        for x, step, off in zip(rolled_curves, offset_steps, offset_fracs):
            if cfg.add_noise_std > 0:
                x = x + rng.normal(0.0, cfg.add_noise_std, size=x.shape)
            X_list.append(x.astype(np.float32))
            y_list.append([pars["EC"], pars["EL"], pars["EJ"], pars["g"], pars["fr"], off])
            source_list.append("joint")
            offset_step_list.append(step)

        done += 1
        if done % 10 == 0 or done == total_base:
            print(f"[{done}/{total_base}] elapsed = {time.time() - t0:.1f} s")

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.float32)
    source_arr = np.asarray(source_list)
    offset_steps = np.asarray(offset_step_list, dtype=np.int32)

    np.savez_compressed(
        cfg.save_path,
        X=X,
        y=y,
        source_type=source_arr,
        offset_steps=offset_steps,
        flux_base_periods=flux_base,
        center_params=np.array([cfg.EC0, cfg.EL0, cfg.EJ0, cfg.g0, cfg.fr0], dtype=np.float32),
        EC_span=np.float32(cfg.EC_span),
        EL_span=np.float32(cfg.EL_span),
        EJ_span=np.float32(cfg.EJ_span),
        g_span=np.float32(cfg.g_span),
        fr_span=np.float32(cfg.fr_span),
        n_axis_points=np.int32(cfg.n_axis_points),
        n_joint_random=np.int32(cfg.n_joint_random),
        n_offset_steps=np.int32(cfg.n_offset_steps),
        n_flux_points=np.int32(cfg.n_flux_points),
        flux_periods=np.float32(cfg.flux_periods),
    )

    elapsed = time.time() - t0
    print(f"\nSaved dataset to: {cfg.save_path}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Elapsed time: {elapsed:.1f} s")


@dataclass
class TrainConfig:
    data_path: str = "fluxonium_better_local_dataset.npz"
    model_save_path: str = "fluxonium_inverse_cnn_better.pt"

    batch_size: int = 128
    num_epochs: int = 120
    lr: float = 1e-3
    weight_decay: float = 1e-5

    train_frac: float = 0.8
    val_frac: float = 0.1
    test_frac: float = 0.1

    use_offset_sincos: bool = True
    normalize_targets: bool = True

    hidden_dim: int = 128
    seed: int = 1234
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class FluxoniumDataset(Dataset):
    def __init__(
        self,
        data_path: str,
        use_offset_sincos: bool = True,
        normalize_targets: bool = True,
        target_mean: Optional[np.ndarray] = None,
        target_std: Optional[np.ndarray] = None,
    ):
        data = np.load(data_path, allow_pickle=True)

        X = data["X"].astype(np.float32)
        y_raw = data["y"].astype(np.float32)  # [EC, EL, EJ, g, fr, offset_fraction]

        if y_raw.shape[1] != 6:
            raise ValueError(f"Expected y shape (N, 6), got {y_raw.shape}")

        params = y_raw[:, :5]
        offset_frac = y_raw[:, 5]

        if use_offset_sincos:
            theta = 2.0 * np.pi * offset_frac
            offset_target = np.stack([np.cos(theta), np.sin(theta)], axis=1).astype(np.float32)
            y = np.concatenate([params, offset_target], axis=1).astype(np.float32)
        else:
            y = y_raw.copy()

        # 3 channels: raw, normalized, derivative(normalized)
        X_raw = X.astype(np.float32)
        x_mean = X.mean(axis=1, keepdims=True)
        x_std = X.std(axis=1, keepdims=True)
        x_std = np.where(x_std < 1e-8, 1.0, x_std)
        X_norm = ((X - x_mean) / x_std).astype(np.float32)
        dX = np.gradient(X_norm, axis=1).astype(np.float32)
        X_final = np.stack([X_raw, X_norm, dX], axis=1).astype(np.float32)

        self.X = X_final
        self.y = y
        self.use_offset_sincos = use_offset_sincos
        self.normalize_targets = normalize_targets

        if normalize_targets:
            if target_mean is None or target_std is None:
                self.target_mean = self.y.mean(axis=0).astype(np.float32)
                self.target_std = self.y.std(axis=0).astype(np.float32)
                self.target_std = np.where(self.target_std < 1e-8, 1.0, self.target_std)
            else:
                self.target_mean = np.asarray(target_mean, dtype=np.float32)
                self.target_std = np.asarray(target_std, dtype=np.float32)
                self.target_std = np.where(self.target_std < 1e-8, 1.0, self.target_std)

            self.y = ((self.y - self.target_mean) / self.target_std).astype(np.float32)
        else:
            self.target_mean = None
            self.target_std = None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])


class ConvBlock(nn.Module):
    def __init__(self, c_in: int, c_out: int, k: int = 5, p: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c_in, c_out, kernel_size=k, padding=k // 2),
            nn.BatchNorm1d(c_out),
            nn.GELU(),
            nn.Conv1d(c_out, c_out, kernel_size=k, padding=k // 2),
            nn.BatchNorm1d(c_out),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Dropout(p),
        )

    def forward(self, x):
        return self.net(x)


class FluxoniumInverseCNN(nn.Module):
    def __init__(self, in_channels: int, out_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 32, k=7, p=0.05),
            ConvBlock(32, 64, k=5, p=0.08),
            ConvBlock(64, 128, k=5, p=0.10),
            ConvBlock(128, hidden_dim, k=3, p=0.12),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        x = self.features(x)
        return self.head(x)


class WeightedMSELoss(nn.Module):
    def __init__(self, weights):
        super().__init__()
        self.register_buffer("weights", torch.tensor(weights, dtype=torch.float32))

    def forward(self, pred, target):
        return torch.mean(self.weights * (pred - target) ** 2)


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def split_lengths(n: int, train_frac: float, val_frac: float, test_frac: float) -> Tuple[int, int, int]:
    if not math.isclose(train_frac + val_frac + test_frac, 1.0, rel_tol=1e-6, abs_tol=1e-6):
        raise ValueError("train_frac + val_frac + test_frac must equal 1.0")
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    n_test = n - n_train - n_val
    return n_train, n_val, n_test


def build_datasets(cfg: TrainConfig):
    full_ds_for_stats = FluxoniumDataset(
        data_path=cfg.data_path,
        use_offset_sincos=cfg.use_offset_sincos,
        normalize_targets=False,
    )

    y_all = full_ds_for_stats.y
    if cfg.normalize_targets:
        target_mean = y_all.mean(axis=0).astype(np.float32)
        target_std = y_all.std(axis=0).astype(np.float32)
        target_std = np.where(target_std < 1e-8, 1.0, target_std)
    else:
        target_mean, target_std = None, None

    full_ds = FluxoniumDataset(
        data_path=cfg.data_path,
        use_offset_sincos=cfg.use_offset_sincos,
        normalize_targets=cfg.normalize_targets,
        target_mean=target_mean,
        target_std=target_std,
    )

    n = len(full_ds)
    n_train, n_val, n_test = split_lengths(n, cfg.train_frac, cfg.val_frac, cfg.test_frac)
    generator = torch.Generator().manual_seed(cfg.seed)
    train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test], generator=generator)
    return full_ds, train_ds, val_ds, test_ds, target_mean, target_std


def make_loaders(cfg: TrainConfig, train_ds, val_ds, test_ds):
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader


def denormalize_targets(y_pred: torch.Tensor, y_true: torch.Tensor, mean, std, device):
    if mean is None or std is None:
        return y_pred, y_true
    mean_t = torch.tensor(mean, dtype=torch.float32, device=device)
    std_t = torch.tensor(std, dtype=torch.float32, device=device)
    return y_pred * std_t + mean_t, y_true * std_t + mean_t


def evaluate_model(model, loader, criterion, device, target_mean=None, target_std=None):
    model.eval()
    total_loss = 0.0
    n_total = 0
    preds_denorm = []
    trues_denorm = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)

            bs = xb.size(0)
            total_loss += loss.item() * bs
            n_total += bs

            pred_d, yb_d = denormalize_targets(pred, yb, target_mean, target_std, device)
            preds_denorm.append(pred_d.cpu().numpy())
            trues_denorm.append(yb_d.cpu().numpy())

    avg_loss = total_loss / max(n_total, 1)
    preds_denorm = np.concatenate(preds_denorm, axis=0)
    trues_denorm = np.concatenate(trues_denorm, axis=0)
    mae = np.mean(np.abs(preds_denorm - trues_denorm), axis=0)
    return avg_loss, mae, preds_denorm, trues_denorm


def pretty_metric_names(use_offset_sincos: bool):
    if use_offset_sincos:
        return ["EC", "EL", "EJ", "g", "fr", "offset_cos", "offset_sin"]
    return ["EC", "EL", "EJ", "g", "fr", "offset_fraction"]


def train(cfg: TrainConfig):
    set_seed(cfg.seed)
    os.makedirs(os.path.dirname(cfg.model_save_path) or ".", exist_ok=True)

    full_ds, train_ds, val_ds, test_ds, target_mean, target_std = build_datasets(cfg)
    train_loader, val_loader, test_loader = make_loaders(cfg, train_ds, val_ds, test_ds)

    sample_x, sample_y = full_ds[0]
    in_channels = sample_x.shape[0]
    out_dim = sample_y.shape[0]

    model = FluxoniumInverseCNN(in_channels=in_channels, out_dim=out_dim, hidden_dim=cfg.hidden_dim).to(cfg.device)

    criterion = WeightedMSELoss([1.0, 1.0, 1.0, 1.0, 2.0, 1.5, 1.5]).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    best_val_loss = float("inf")
    best_state = None

    print(f"Device: {cfg.device}")
    print(f"Dataset sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")
    print(f"Input channels: {in_channels}, Output dim: {out_dim}")

    for epoch in range(1, cfg.num_epochs + 1):
        model.train()
        running_loss = 0.0
        n_total = 0

        for xb, yb in train_loader:
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            bs = xb.size(0)
            running_loss += loss.item() * bs
            n_total += bs

        train_loss = running_loss / max(n_total, 1)
        val_loss, val_mae, _, _ = evaluate_model(model, val_loader, criterion, cfg.device, target_mean, target_std)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                "model_state_dict": model.state_dict(),
                "config": cfg.__dict__,
                "target_mean": target_mean.tolist() if target_mean is not None else None,
                "target_std": target_std.tolist() if target_std is not None else None,
                "metric_names": pretty_metric_names(cfg.use_offset_sincos),
            }
            torch.save(best_state, cfg.model_save_path)

        if epoch == 1 or epoch % 10 == 0 or epoch == cfg.num_epochs:
            metric_names = pretty_metric_names(cfg.use_offset_sincos)
            mae_str = ", ".join([f"{n}={m:.5f}" for n, m in zip(metric_names, val_mae)])
            print(
                f"Epoch {epoch:03d} | train_loss={train_loss:.6f} | "
                f"val_loss={val_loss:.6f} | val_MAE: {mae_str}"
            )

    print(f"\nBest model saved to: {cfg.model_save_path}")
    print(f"Best val loss: {best_val_loss:.6f}")

    model.load_state_dict(best_state["model_state_dict"])
    test_loss, test_mae, preds_denorm, trues_denorm = evaluate_model(
        model, test_loader, criterion, cfg.device, target_mean, target_std
    )

    metric_names = pretty_metric_names(cfg.use_offset_sincos)
    print("\nTest results:")
    print(f"  test_loss = {test_loss:.6f}")
    for n, m in zip(metric_names, test_mae):
        print(f"  {n}_MAE = {m:.6f}")

    print("\nA few prediction examples from test set:")
    n_show = min(5, len(preds_denorm))
    for i in range(n_show):
        print(f"Example {i + 1}:")
        print("  pred:", np.round(preds_denorm[i], 6))
        print("  true:", np.round(trues_denorm[i], 6))


def preprocess_curve_for_model(curve_1d: np.ndarray) -> np.ndarray:
    curve = curve_1d.astype(np.float32)
    c_mean = curve.mean(keepdims=True)
    c_std = curve.std(keepdims=True)
    c_std = np.where(c_std < 1e-8, 1.0, c_std)
    x_norm = (curve - c_mean) / c_std
    dx = np.gradient(x_norm).astype(np.float32)
    x = np.stack([curve, x_norm, dx], axis=0)[None, ...]
    return x


def predict_single_curve(curve_1d: np.ndarray, checkpoint_path: str):
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    cfg_dict = ckpt["config"]
    metric_names = ckpt["metric_names"]
    target_mean = np.asarray(ckpt["target_mean"], dtype=np.float32) if ckpt["target_mean"] is not None else None
    target_std = np.asarray(ckpt["target_std"], dtype=np.float32) if ckpt["target_std"] is not None else None

    x = preprocess_curve_for_model(curve_1d)
    x_t = torch.tensor(x, dtype=torch.float32)

    model = FluxoniumInverseCNN(
        in_channels=x.shape[1],
        out_dim=len(metric_names),
        hidden_dim=cfg_dict["hidden_dim"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    with torch.no_grad():
        pred = model(x_t).cpu().numpy()[0]

    if target_mean is not None and target_std is not None:
        pred = pred * target_std + target_mean

    result = {name: float(val) for name, val in zip(metric_names, pred)}

    if "offset_cos" in result and "offset_sin" in result:
        angle = math.atan2(result["offset_sin"], result["offset_cos"])
        if angle < 0:
            angle += 2 * math.pi
        result["offset_fraction_recovered"] = angle / (2 * math.pi)

    return result


def best_circular_shift(curve_ref: np.ndarray, curve_meas: np.ndarray):
    a = curve_ref - np.mean(curve_ref)
    b = curve_meas - np.mean(curve_meas)
    corr = np.fft.ifft(np.fft.fft(a) * np.conj(np.fft.fft(b)))
    corr = np.real(corr)
    best_shift = int(np.argmax(corr))
    offset_fraction = best_shift / len(curve_ref)
    return best_shift, offset_fraction

In [ ]:
# For 500 training sets, it needs ~2h to finish data generating + model training
# Input guessed params below and the range of fluctuation
# Numbers of points to choose do not have to change if no need of fitting other params besides offset

if __name__ == "__main__":
    gen_cfg = GenerateConfig(
        EC0=1.0,
        EL0=0.8,
        EJ0=3.0,
        g0=0.03,
        fr0=5.1,
        EC_span=0.10,
        EL_span=0.10,
        EJ_span=0.50,
        g_span=1.0,
        fr_span=0.05,
        n_axis_points=16,
        n_joint_random=420,
        n_flux_points=121, # numbers of points pciked in one period can change to smaller value if need a more accurate flux (make sure CNN in "Generating exp..." notebook is same with this)
        flux_periods=1.0,
        flux_start_periods=0.0,
        n_offset_steps=31,
        add_noise_std=0.0,
        q_levels=10,
        N_phot=6,
        grid=2000,
        flux_max=12 * np.pi,
        seed=1234,
        save_path="fluxonium_better_local_dataset.npz", # path you want to save this ML model
    )

    build_better_local_dataset(gen_cfg)

    train_cfg = TrainConfig(
        data_path="fluxonium_better_local_dataset.npz",
        model_save_path="fluxonium_inverse_cnn_better.pt",
        batch_size=128,
        num_epochs=120,
        lr=1e-3,
        weight_decay=1e-5,
        train_frac=0.8,
        val_frac=0.1,
        test_frac=0.1,
        use_offset_sincos=True,
        normalize_targets=True,
        hidden_dim=128,
        seed=1234,
    )

    train(train_cfg)

In [ ]:
# Loading experimental data and use ML model to analyze it
# path of model and experimental data
exp_npz_path = "exp_res_spec_period_for_cnn_test.npz"
model_ckpt_path = "fluxonium_inverse_cnn_better.pt"
readout_lo_hz = 0   # LO freq if your experimental data is ssb freq

import matplotlib.pyplot as plt

data = np.load(exp_npz_path, allow_pickle=True)
flux_resampled = data["flux_resampled"]
fr_resampled_hz = data["fr_resampled"]

print("Loaded experimental test curve:")
print("  flux_resampled shape =", flux_resampled.shape)
print("  fr_resampled_hz shape =", fr_resampled_hz.shape)
print("  fr_resampled_hz mean =", np.mean(fr_resampled_hz))

fr_resampled_ghz = (readout_lo_hz + fr_resampled_hz) / 1e9
print("\nConverted curve stats (GHz):")
print("  min  =", np.min(fr_resampled_ghz))
print("  max  =", np.max(fr_resampled_ghz))
print("  mean =", np.mean(fr_resampled_ghz))

plt.figure(figsize=(7, 4))
plt.plot(flux_resampled, fr_resampled_ghz, "o-", ms=3)
plt.xlabel("Flux / Current")
plt.ylabel("Fitted Readout Frequency (GHz)")
plt.title("Experimental curve sent to 1D CNN")
plt.tight_layout()
plt.show()

pred = predict_single_curve(fr_resampled_ghz, model_ckpt_path)
print("\nPredicted parameters from 1D CNN:")
for k, v in pred.items():
    print(f"  {k:24s} = {v}")

In [ ]:
# Helper fucntions for plotting

def plot_theoretical_fr_vs_flux_with_offset(EC,EL,EJ,fr,g,flux_min=0.0,flux_max=1.0,n_flux_points=121,use_period_units=True,offset=0.0,offset_is_fraction=True,q_levels=10,N_phot=6,grid=2000,flux_max_basis=12*np.pi):
    """
    Plot theoretical readout frequency vs flux, with an added horizontal offset.

    Parameters
    ----------
    flux_min, flux_max:
        If use_period_units=True, these are in period units.
        If False, these are in radians.

    offset:
        Horizontal shift to apply.

    offset_is_fraction:
        Only relevant when use_period_units=True.
        True  -> offset is in units of one period (e.g. 0.25 means pi/2 shift)
        False -> offset is in radians
    """

    flux_vals = np.linspace(flux_min, flux_max, n_flux_points, endpoint=False)

    if use_period_units:
        if offset_is_fraction:
            flux_ext_vals = 2.0 * np.pi * (flux_vals - offset)
        else:
            flux_ext_vals = 2.0 * np.pi * flux_vals - offset
        x_plot = flux_vals
    else:
        if offset_is_fraction:
            flux_ext_vals = flux_vals - 2.0 * np.pi * offset
        else:
            flux_ext_vals = flux_vals - offset
        x_plot = flux_vals

    fr_g_list = []
    fr_e_list = []

    for phi in flux_ext_vals:
        fr_g, fr_e = _readout_frequencies(
            EC=EC,
            EL=EL,
            EJ=EJ,
            flux_ext=phi,
            fr=fr,
            g=g,
            q_levels=q_levels,
            N_phot=N_phot,
            grid=grid,
            flux_max=flux_max_basis,
        )
        fr_g_list.append(fr_g)
        fr_e_list.append(fr_e)

    fr_g_arr = np.asarray(fr_g_list, dtype=float)
    fr_e_arr = np.asarray(fr_e_list, dtype=float)

    return x_plot, fr_g_arr, fr_e_arr

In [ ]:
# Plotting of ML fitting data and real data

readout_lo_hz = 0 # Change to what you need like above

data = np.load(exp_npz_path, allow_pickle=True)
flux_resampled = data["flux_resampled"]
fr_resampled_hz = data["fr_resampled"]
fr_exp_ghz = (readout_lo_hz + fr_resampled_hz) / 1e9

x_theory, fr_theory, _ = plot_theoretical_fr_vs_flux_with_offset(
    EC = pred["EC"],
    EL = pred["EL"],
    EJ = pred["EJ"],
    fr = pred["fr"],
    g = pred["g"],
    flux_min = 0.0,
    flux_max = 1.0,
    n_flux_points = len(fr_exp_ghz),
    use_period_units = True,
    offset = 0.3015187089842214,
    offset_is_fraction = True,
)

flux_norm = (flux_resampled - flux_resampled.min()) / (
    flux_resampled.max() - flux_resampled.min()
)

plt.figure(figsize=(8, 5))
plt.plot(
    flux_norm,
    fr_exp_ghz,
    "o-",
    ms=3,
    color="tab:blue",
    label="Experimental (resampled)",
)

plt.plot(
    x_theory,
    fr_theory,
    "-",
    lw=2,
    color="tab:red",
    label="Theory (fit params)",
)

plt.xlabel("Flux (normalized to 1 period)")
plt.ylabel("Readout Frequency (GHz)")
plt.title("Experimental vs Theoretical Resonator Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()